# BraTS 2024 - Slice Dataset Generation and Export

This notebook handles:
1. Mounting Google Drive and installing the `brain_mri` package.
2. Loading preprocessing configuration and patient splits (`train` / `validate` / `test`).
3. **Single patient sanity check:** validating slice extraction, PyTorch tensors, and visualization.
4. **Full dataset export:** saving all slices to local Colab SSD (`/content/processed`).
5. **Archiving to Google Drive:** packing into a single `processed_slices.zip` file.

## 1. Mount Google Drive and environment setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
from pathlib import Path

repo_path = Path("/content/BraTS_thesis")
branch = "minvalue_norm"

if not repo_path.exists():
    !git clone -b {branch} https://github.com/AdaBro24/BraTS_thesis.git {repo_path}
else:
    !git -C {repo_path} fetch origin
    !git -C {repo_path} checkout {branch}
    !git -C {repo_path} pull origin {branch}

!pip install -e /content/BraTS_thesis

Already on 'minvalue_norm'
Your branch is up to date with 'origin/minvalue_norm'.
From https://github.com/AdaBro24/BraTS_thesis
 * branch            minvalue_norm -> FETCH_HEAD
Already up to date.
Obtaining file:///content/BraTS_thesis
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for brain_mri (pyproject.toml) ... done
  Created wheel for brain_mri: filename=brain_mri-0.1.0-0.editable-py3-none-any.whl size=1304 sha256=0bc1fe27d42d3b007d5638a70ee8a9cfcff86bcd303f336ef3727092ddd0d94d
  Stored in directory: /tmp/pip-ephem-wheel-cache-dh0m5uwx/wheels/4b/5c/05/dacf53db21b47a84a06f984d5251964e1c2594931cea5ebbaa
Successfully built brain_mri
  Attempting uninstall: brain_mri
    Found existing installation: brain_mri 0.1.0
    Uninstalling brain_mri-0.1.0:
      Successfully uninstalled brain_mri-0.1.0


## 2. Data paths and configuration

In [3]:
import sys
from pathlib import Path

# Ensure repository src path is available to the current Python kernel
repo_src = Path("/content/BraTS_thesis/src")
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

import torch

from brain_mri import load_data_config
from brain_mri.data.split import get_or_create_splits
from brain_mri.data.slice_extraction import extract_patient_slices
from brain_mri.data.dataset_export import save_patients_to_pt, export_dataset_split

# Google Drive data paths
ZIP_DIR = Path("/content/drive/MyDrive/engineering_thesis/brats")
DATA_ROOT = ZIP_DIR / "unpacked"
GLI_ROOT = DATA_ROOT / "BraTS-GLI" / "training_data1_v2"

# Load preprocessing YAML config
config = load_data_config(Path("/content/BraTS_thesis/configs/data_config.yaml"))
# Update Z boundaries path to Colab workspace
config.paths.z_boundaries = Path("/content/BraTS_thesis/data/patients_z_boundaries.json")

# Load or generate patient splits (train: 70%, val: 15%, test: 15%)
splits_json_path = ZIP_DIR / "data_splits.json"
splits = get_or_create_splits(folder_path=GLI_ROOT, json_path=splits_json_path)

print("Patient count:")
print(f" - Train:      {len(splits['train'])}")
print(f" - Validation: {len(splits['validate'])}")
print(f" - Test:       {len(splits['test'])}")

Patient count:
 - Train:      944
 - Validation: 203
 - Test:       203


## 3. Single patient test (Sanity check and visualization)

In [ ]:
import matplotlib.pyplot as plt

# Select the first training patient for testing
test_patient_id = splits["train"][0]
test_patient_dir = GLI_ROOT / test_patient_id
print(f"Testing patient: {test_patient_id}")

# Extract slices
slices = extract_patient_slices(test_patient_dir, config=config)
print(f"Extracted {len(slices)} slices.")

# Save slices to a temporary test directory
test_output_dir = Path("/content/test_processed")
saved_files = save_patients_to_pt(slices, test_output_dir)
print(f"Saved {len(saved_files)} .pt files to {test_output_dir}")

# Inspect a middle slice (where tumor is usually present)
sample_file = saved_files[len(saved_files) // 2]
data = torch.load(sample_file, weights_only=True)

img = data["image"]  # (4, 224, 224)
mask = data["mask"]  # (224, 224)

print("\n--- Tensor verification ---")
print(f"Image shape:       {img.shape}, dtype: {img.dtype}")
print(f"Mask shape:        {mask.shape}, dtype: {mask.dtype}")
print(f"Unique mask labels: {torch.unique(mask).tolist()}")
print(f"Background value (corner): {img[:, 0, 0].tolist()}")

# Plot all 4 MRI modalities + segmentation mask
modalities = ["T1", "T1CE", "T2", "FLAIR"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i in range(4):
    axes[i].imshow(img[i].numpy(), cmap="gray")
    axes[i].set_title(modalities[i])
    axes[i].axis("off")

axes[4].imshow(mask.numpy(), cmap="tab10", vmin=0, vmax=4)
axes[4].set_title(f"Mask (z={data['z_idx']})")
axes[4].axis("off")

plt.tight_layout()
plt.show()

## 4. Full dataset export (to fast local Colab SSD)

Writing to `/content/processed` uses the fast local NVMe SSD (avoids slow Google Drive network latency and timeouts).

In [4]:
# Export to fast local Colab SSD (super fast!)
PROCESSED_ROOT = Path("/content/processed")

stats = export_dataset_split(
    data_root=GLI_ROOT,
    splits=splits,
    output_root=PROCESSED_ROOT,
    config=config,
)
print("Export summary:", stats)

Exporting train:   0%|          | 0/944 [00:00<?, ?patient/s]

: 

## 5. Archive to Google Drive as a single ZIP file

Packing all slices into a single zip archive on Google Drive takes only 1-2 minutes and prevents Colab timeouts.

In [ ]:
zip_target = ZIP_DIR / "processed_slices.zip"
print(f"Creating zip archive at {zip_target}...")
!zip -r -q {zip_target} /content/processed
print("Archive created successfully on Google Drive!")